## OWLv2 訓練 (Colab)

ローカル収集済みの `PhysicalAI_adv/data/OWL` を読む。zip は展開せず中身を直接開く。チェックポイントがあれば同じセルを再実行して再開する。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q transformers accelerate scipy pillow

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/content/drive/MyDrive/PhysicalAI_adv/dog")
DATA_DIR = Path("/content/drive/MyDrive/PhysicalAI_adv/data/OWL")
DETECT_DIR = Path("/content/drive/MyDrive/PhysicalAI_adv/detect_models")
ZIP_CACHE = Path("/content/owl_zips")

os.makedirs(DETECT_DIR, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print(REPO)
print(DATA_DIR)
print(DETECT_DIR)
print(ZIP_CACHE)

## 学習

Drive の zip を `/content/owl_zips` にコピーしてから中を直接読む。ばらファイルも読む。初回に `sample_order.json` でシャッフル順を固定し、checkpoint ごとに `train_progress.json` へ見たサンプルを書く。切れても同じセルを再実行すれば、同じ順番の続きから再開する。

In [ ]:
import numpy as np


def _label_map(seg: np.ndarray) -> np.ndarray:
    seg = np.ascontiguousarray(seg)
    if seg.ndim == 3 and seg.shape[-1] in (3, 4):
        rgb = seg[..., :3].astype(np.uint32, copy=False)
        return (rgb[..., 0] << 16) | (rgb[..., 1] << 8) | rgb[..., 2]
    if seg.ndim == 2:
        return seg
    raise ValueError(f"expected HxW or HxWx3 segmentation, got shape {seg.shape}")


def bboxes_from_seg(seg: np.ndarray, ignore_ids=(0,), min_area: int = 1) -> list[dict]:
    labels = _label_map(seg)
    ignore = {int(i) for i in ignore_ids}
    boxes = []
    for label in np.unique(labels):
        label = int(label)
        if label in ignore:
            continue
        ys, xs = np.nonzero(labels == label)
        if ys.size < min_area:
            continue
        x_min = int(xs.min())
        y_min = int(ys.min())
        x_max = int(xs.max())
        y_max = int(ys.max())
        boxes.append(
            {
                "id": label,
                "x_min": x_min,
                "y_min": y_min,
                "x_max": x_max,
                "y_max": y_max,
                "width": x_max - x_min + 1,
                "height": y_max - y_min + 1,
                "area": int(ys.size),
            }
        )
    boxes.sort(key=lambda b: b["area"], reverse=True)
    return boxes

In [ ]:
TRAIN_NAME = "owlv2_thefirststep"
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
DATA_SEED = 42
MODEL_ID = "google/owlv2-base-patch16-ensemble"
BATCH_SIZE = 8
GRAD_ACCUM = 2
SAVE_STEPS = 100

import io
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.optimize import linear_sum_assignment
from torch.utils.data import Dataset
from torchvision.ops import box_convert, generalized_box_iou
from transformers import Owlv2ForObjectDetection, Owlv2Processor, Trainer, TrainerCallback, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

from utils.seg_bbox import bboxes_from_seg

OUTPUT_DIR = DETECT_DIR / TRAIN_NAME
ORDER_PATH = OUTPUT_DIR / "sample_order.json"
PROGRESS_PATH = OUTPUT_DIR / "train_progress.json"
QUERIES = [
    "a yellow ball",
    "a red ball",
    "an orange ball",
    "a pink ball",
    "a purple ball",
    "a blue ball",
    "a cyan ball",
    "a green ball",
    "a white ball",
    "a black ball",
    "a yellow inflatable arch",
    "a red inflatable arch",
    "an orange inflatable arch",
    "a pink inflatable arch",
    "a purple inflatable arch",
    "a blue inflatable arch",
    "a cyan inflatable arch",
    "a green inflatable arch",
    "a white inflatable arch",
    "a black inflatable arch",
]
QUERY_TO_ID = {query: i for i, query in enumerate(QUERIES)}
TRAIN_CONFIG = {
    "train_name": TRAIN_NAME,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "data_seed": DATA_SEED,
    "model_id": MODEL_ID,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
}


def stage_zips(data_dir: Path, cache_dir: Path) -> list[Path]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    staged = []
    for src in sorted(data_dir.glob("*.zip")):
        dest = cache_dir / src.name
        if not dest.exists() or dest.stat().st_size != src.stat().st_size:
            print(f"copy {src.name}")
            shutil.copy2(src, dest)
        staged.append(dest)
    return staged


def load_samples(data_dir: Path) -> list[dict]:
    samples = {}
    for meta_path in data_dir.glob("*.json"):
        rgb_path = meta_path.with_suffix(".png")
        seg_path = meta_path.with_name(meta_path.stem + "_seg.npy")
        if rgb_path.exists() and seg_path.exists():
            samples[meta_path.name] = {
                "kind": "file",
                "key": meta_path.name,
                "meta": meta_path,
                "rgb": rgb_path,
                "seg": seg_path,
            }
    for zip_path in stage_zips(data_dir, ZIP_CACHE):
        with zipfile.ZipFile(zip_path) as archive:
            names = set(archive.namelist())
        for name in names:
            if not name.endswith(".json"):
                continue
            stem = name[:-5]
            png_name = f"{stem}.png"
            npy_name = f"{stem}_seg.npy"
            if png_name in names and npy_name in names:
                samples[name] = {"kind": "zip", "key": name, "zip": zip_path, "stem": stem}
    items = list(samples.values())
    items.sort(key=lambda item: item["key"])
    n_zip = sum(item["kind"] == "zip" for item in items)
    n_file = len(items) - n_zip
    print(f"indexed={len(items)} zip={n_zip} files={n_file}")
    return items


def freeze_order(samples: list[dict]) -> list[dict]:
    by_key = {item["key"]: item for item in samples}
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if ORDER_PATH.exists():
        order = json.loads(ORDER_PATH.read_text())
        ordered = [by_key[key] for key in order if key in by_key]
        missing = [key for key in order if key not in by_key]
        extra_keys = sorted(key for key in by_key if key not in set(order))
        extra = [by_key[key] for key in extra_keys]
        rng = random.Random(DATA_SEED)
        rng.shuffle(extra)
        ordered.extend(extra)
        if extra_keys or missing:
            print(f"order: kept={len(ordered) - len(extra)} new={len(extra)} missing={len(missing)}")
        ORDER_PATH.write_text(json.dumps([item["key"] for item in ordered], indent=2))
        return ordered
    rng = random.Random(DATA_SEED)
    rng.shuffle(samples)
    ORDER_PATH.write_text(json.dumps([item["key"] for item in samples], indent=2))
    print(f"order: wrote {len(samples)} keys to {ORDER_PATH.name}")
    return samples


class SaveDataProgress(TrainerCallback):
    def __init__(self, keys: list[str]):
        self.keys = keys

    def _write(self, state):
        seen = min(len(self.keys), int(state.global_step) * BATCH_SIZE * GRAD_ACCUM)
        PROGRESS_PATH.write_text(
            json.dumps(
                {
                    "global_step": int(state.global_step),
                    "next_index": seen,
                    "n_total": len(self.keys),
                    "last_seen_key": self.keys[seen - 1] if seen else None,
                    "seen_keys": self.keys[:seen],
                },
                indent=2,
            )
        )
        print(f"progress step={state.global_step} seen={seen}/{len(self.keys)}")

    def on_save(self, args, state, control, **kwargs):
        self._write(state)

    def on_train_end(self, args, state, control, **kwargs):
        self._write(state)


def xyxy_to_cxcywh(box, width, height):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(x2 - x1 + 1.0, 1.0)
    bh = max(y2 - y1 + 1.0, 1.0)
    cx = (x1 + x2) * 0.5 / width
    cy = (y1 + y2) * 0.5 / height
    return [cx, cy, bw / width, bh / height]


def labels_from_seg(seg, meta: dict, width: int, height: int):
    idx_to_query = {int(obj["idx"]): obj["query"] for obj in meta["objects"]}
    boxes, classes = [], []
    for box in bboxes_from_seg(seg):
        query = idx_to_query.get(int(box["id"]) - 1)
        if query not in QUERY_TO_ID:
            continue
        boxes.append(xyxy_to_cxcywh([box["x_min"], box["y_min"], box["x_max"], box["y_max"]], width, height))
        classes.append(QUERY_TO_ID[query])
    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
        "class_labels": torch.tensor(classes, dtype=torch.int64),
    }


class OwlDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
        self._zips = {}

    def __len__(self):
        return len(self.samples)

    def _archive(self, path: Path):
        archive = self._zips.get(path)
        if archive is None:
            archive = zipfile.ZipFile(path)
            self._zips[path] = archive
        return archive

    def __getitem__(self, idx):
        item = self.samples[idx]
        if item["kind"] == "zip":
            archive = self._archive(item["zip"])
            stem = item["stem"]
            image = Image.open(archive.open(f"{stem}.png")).convert("RGB")
            meta = json.loads(archive.read(f"{stem}.json"))
            seg = np.load(io.BytesIO(archive.read(f"{stem}_seg.npy")))
        else:
            image = Image.open(item["rgb"]).convert("RGB")
            meta = json.loads(item["meta"].read_text())
            seg = np.load(item["seg"])
        return {"image": image, "labels": labels_from_seg(seg, meta, image.width, image.height)}


def collate_fn(batch):
    images = [ex["image"] for ex in batch]
    labels = [ex["labels"] for ex in batch]
    encoded = processor(text=[QUERIES] * len(batch), images=images, return_tensors="pt")
    encoded["labels"] = labels
    return encoded


def detection_loss(outputs, labels):
    logits = outputs.logits
    pred_boxes = outputs.pred_boxes
    obj_logits = outputs.objectness_logits
    if obj_logits.ndim == 3:
        obj_logits = obj_logits.squeeze(-1)
    losses = []
    for i, target in enumerate(labels):
        gt_boxes = target["boxes"].to(pred_boxes.device)
        gt_cls = target["class_labels"].to(pred_boxes.device)
        pred_b = pred_boxes[i]
        pred_l = logits[i]
        pred_o = obj_logits[i]
        obj_target = torch.zeros_like(pred_o)
        if gt_boxes.numel() == 0:
            losses.append(F.binary_cross_entropy_with_logits(pred_o, obj_target))
            continue
        with torch.no_grad():
            cost_class = -pred_l.softmax(-1)[:, gt_cls]
            cost_bbox = torch.cdist(pred_b, gt_boxes, p=1)
            pred_xyxy = box_convert(pred_b, in_fmt="cxcywh", out_fmt="xyxy").clamp(0, 1)
            gt_xyxy = box_convert(gt_boxes, in_fmt="cxcywh", out_fmt="xyxy").clamp(0, 1)
            cost_giou = -generalized_box_iou(pred_xyxy, gt_xyxy)
            cost = (2.0 * cost_class + 5.0 * cost_bbox + 2.0 * cost_giou).detach().cpu().numpy()
            cost = np.nan_to_num(cost, nan=1e5, posinf=1e5, neginf=1e5)
            pred_idx, gt_idx = linear_sum_assignment(cost)
        pred_idx = torch.as_tensor(pred_idx, device=pred_b.device)
        gt_idx = torch.as_tensor(gt_idx, device=pred_b.device)
        obj_target[pred_idx] = 1.0
        cls_loss = F.cross_entropy(pred_l[pred_idx], gt_cls[gt_idx])
        box_l1 = F.l1_loss(pred_b[pred_idx], gt_boxes[gt_idx])
        giou = 1.0 - torch.diag(
            generalized_box_iou(
                box_convert(pred_b[pred_idx], in_fmt="cxcywh", out_fmt="xyxy").clamp(0, 1),
                box_convert(gt_boxes[gt_idx], in_fmt="cxcywh", out_fmt="xyxy").clamp(0, 1),
            )
        ).mean()
        obj_loss = F.binary_cross_entropy_with_logits(pred_o, obj_target)
        losses.append(cls_loss + 5.0 * box_l1 + 2.0 * giou + obj_loss)
    return torch.stack(losses).mean()


class Owlv2ForTraining(Owlv2ForObjectDetection):
    def forward(self, input_ids=None, pixel_values=None, attention_mask=None, labels=None, **kwargs):
        outputs = super().forward(
            input_ids=input_ids,
            pixel_values=pixel_values,
            attention_mask=attention_mask,
            **kwargs,
        )
        if labels is not None:
            outputs.loss = detection_loss(outputs, labels)
        return outputs


samples = freeze_order(load_samples(DATA_DIR))
if not samples:
    raise FileNotFoundError(f"no rgb/seg pairs under {DATA_DIR}")
print(f"queries={len(QUERIES)} samples={len(samples)}")
if PROGRESS_PATH.exists():
    prev = json.loads(PROGRESS_PATH.read_text())
    print(f"recorded seen={prev.get('next_index')} last={prev.get('last_seen_key')}")

processor = Owlv2Processor.from_pretrained(MODEL_ID)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train_config.json").write_text(json.dumps(TRAIN_CONFIG, indent=2))

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=10,
    fp16=True,
    seed=DATA_SEED,
    data_seed=DATA_SEED,
    remove_unused_columns=False,
    report_to=[],
    dataloader_num_workers=0,
    ignore_data_skip=False,
)
last_ckpt = get_last_checkpoint(str(OUTPUT_DIR))
print(f"resume_from={last_ckpt}")
model = Owlv2ForTraining.from_pretrained(MODEL_ID)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=OwlDataset(samples),
    data_collator=collate_fn,
    processing_class=processor,
    callbacks=[SaveDataProgress([item["key"] for item in samples])],
)
trainer.train(resume_from_checkpoint=last_ckpt)
trainer.save_model(str(OUTPUT_DIR / "final"))
trainer.save_state()
TRAIN_CONFIG["global_step"] = int(trainer.state.global_step)
TRAIN_CONFIG["resume_from"] = last_ckpt
(OUTPUT_DIR / "train_config.json").write_text(json.dumps(TRAIN_CONFIG, indent=2))
print(f"saved {OUTPUT_DIR} step={TRAIN_CONFIG['global_step']}")